In [ ]:
# Load necessary libraries
import sys
import os
import anndata as ad
import pandas as pd
import scanpy as sc
import seaborn as sns; sns.set(color_codes=True)
import collections
import re
from matplotlib import pyplot as plt
import random
import numpy as np
from collections import OrderedDict
import copy

random.seed(10)



In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

In [ ]:
sc.logging.print_header()

In [ ]:
imageIDMapQP = {'10202021_BaharehNP_2997-ADCortex_EDTA.qptiff - resolution #1':'2997',
       '11162021_Bahareh-3155_EDTA.qptiff - resolution #1':'3155',
       '10122021_BaharehNP_ADCortex_EDTA.qptiff - resolution #1':'3196',
       '12032021-Bahareh_3280_Alz.qptiff - resolution #1':'3280',
       '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6.qptiff - resolution #1':'1796',
       '12252021_Bahareh-3628-Cntr.qptiff - resolution #1':'3628',
       '11092021_BaharehNP_3026CtlCorte.qptiff - resolution #1':'3026',
       '10222021_BaharehNP_1873_CtrlCortex.qptiff - resolution #1':'1873'}
imageTypeMapQP = {'10202021_BaharehNP_2997-ADCortex_EDTA.qptiff - resolution #1':'AD',
       '11162021_Bahareh-3155_EDTA.qptiff - resolution #1':'AD',
       '10122021_BaharehNP_ADCortex_EDTA.qptiff - resolution #1':'AD',
       '12032021-Bahareh_3280_Alz.qptiff - resolution #1':'AD',
       '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6.qptiff - resolution #1':'Ctl',
       '12252021_Bahareh-3628-Cntr.qptiff - resolution #1':'Ctl',
       '11092021_BaharehNP_3026CtlCorte.qptiff - resolution #1':'Ctl',
       '10222021_BaharehNP_1873_CtrlCortex.qptiff - resolution #1':'Ctl'}

# generate metadata for all cells

## load in sample csv files containing coordinates

In [ ]:

# load in samples per cell and collect coordinates

folder_path = 'marker_intensities/'


# IBA

iba_csvs = [
'10222021_BaharehNP_1873_CtrlCortex_IBA1_cell1.csv',
 '11162021_Bahareh-3155_EDTA_IBA1_cell1.csv',
 '10202021_BaharehNP_2997-ADCortex_EDTA_IBA1_cell1.csv',
 '12252021_Bahareh-3628-Cntr_IBA1_cell1.csv',
 '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_IBA1_cell1.csv',
 '12032021-Bahareh_3280_Alz_IBA1_cell1.csv',
 '10122021_BaharehNP_ADCortex_EDTA_IBA1_cell1.csv',
 '11092021_BaharehNP_3026CtlCorte_IBA1_cell1.csv'
]

iba_sample_num_dict = {
    '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_IBA1_cell1.csv': "1796",
    '10122021_BaharehNP_ADCortex_EDTA_IBA1_cell1.csv': "3196",
    '10202021_BaharehNP_2997-ADCortex_EDTA_IBA1_cell1.csv': "2997",
    '10222021_BaharehNP_1873_CtrlCortex_IBA1_cell1.csv': "1873",
    '11092021_BaharehNP_3026CtlCorte_IBA1_cell1.csv': "3026",
    '11162021_Bahareh-3155_EDTA_IBA1_cell1.csv': "3155",
    '12032021-Bahareh_3280_Alz_IBA1_cell1.csv': "3280",
    '12252021_Bahareh-3628-Cntr_IBA1_cell1.csv': "3628",
}
iba_sample_num_dict = {key: [value, 'IBA1'] for key, value in iba_sample_num_dict.items()}


# GFAP

gfap_csvs = [
    "10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_GFAP_cell.csv",
    "10122021_BaharehNP_ADCortex_EDTA_GFAP_cell.csv",
    "10202021_BaharehNP_2997-ADCortex_EDTA_GFAP_cell.csv",
    "10222021_BaharehNP_1873_CtrlCortex_GFAP_cell.csv",
    "11092021_BaharehNP_3026CtlCorte_GFAP_cell.csv",
    "11162021_Bahareh-3155_EDTA_GFAP_cell.csv",
    "12032021-Bahareh_3280_Alz_H2A.x_cell.csv",
    "12252021_Bahareh-3628-Cntr_H2A.x_cell.csv"   
]

gfap_sample_num_dict = {
    "10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_GFAP_cell.csv":"1796",
    "10122021_BaharehNP_ADCortex_EDTA_GFAP_cell.csv":"3196",
    "10202021_BaharehNP_2997-ADCortex_EDTA_GFAP_cell.csv":"2997",
    "10222021_BaharehNP_1873_CtrlCortex_GFAP_cell.csv":"1873",
    "11092021_BaharehNP_3026CtlCorte_GFAP_cell.csv":"3026",
    "11162021_Bahareh-3155_EDTA_GFAP_cell.csv":"3155",
    "12032021-Bahareh_3280_Alz_H2A.x_cell.csv":"3280",
    "12252021_Bahareh-3628-Cntr_H2A.x_cell.csv":"3628"
}
gfap_sample_num_dict = {key: [value, 'GFAP'] for key, value in gfap_sample_num_dict.items()}


# Olig2

olig2_csvs = [
    "11092021_BaharehNP_3026CtlCorte_Olig2_cell_updated.csv",
    "11162021_Bahareh-3155_EDTA_Olig2_cell.csv",
    "12032021-Bahareh_3280_Alz_Olig2_cell.csv",
    "12252021_Bahareh-3628-Cntr_Olig2_cell.csv"
]

olig2_sample_num_dict = {
    "11092021_BaharehNP_3026CtlCorte_Olig2_cell_updated.csv":"3026",
    "11162021_Bahareh-3155_EDTA_Olig2_cell.csv":"3155",
    "12032021-Bahareh_3280_Alz_Olig2_cell.csv":"3280",
    "12252021_Bahareh-3628-Cntr_Olig2_cell.csv":"3628"
}
olig2_sample_num_dict = {key: [value, 'Olig2'] for key, value in olig2_sample_num_dict.items()}



# NeuN

neun_csvs = [
'10222021_BaharehNP_1873_CtrlCortex_NeuN_cell.csv',
 '11162021_Bahareh-3155_EDTA_NeuN_cell.csv',
 '10202021_BaharehNP_2997-ADCortex_EDTA_NeuN_cell.csv',
 '12252021_Bahareh-3628-Cntr_NeuN_cell.csv',
 '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_NeuN_cell.csv',
 '12032021-Bahareh_3280_Alz_NeuN_cell.csv',
 '10122021_BaharehNP_ADCortex_EDTA_NeuN_cell.csv',
 '11092021_BaharehNP_3026CtlCorte_NeuN_cell.csv'
]

neun_sample_num_dict = {
    '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_NeuN_cell.csv': "1796",
    '10122021_BaharehNP_ADCortex_EDTA_NeuN_cell.csv': "3196",
    '10202021_BaharehNP_2997-ADCortex_EDTA_NeuN_cell.csv': "2997",
    '10222021_BaharehNP_1873_CtrlCortex_NeuN_cell.csv': "1873",
    '11092021_BaharehNP_3026CtlCorte_NeuN_cell.csv': "3026",
    '11162021_Bahareh-3155_EDTA_NeuN_cell.csv': "3155",
    '12032021-Bahareh_3280_Alz_NeuN_cell.csv': "3280",
    '12252021_Bahareh-3628-Cntr_NeuN_cell.csv': "3628"
}

neun_sample_num_dict = {key: [value, 'NeuN'] for key, value in neun_sample_num_dict.items()}



# Abeta

abeta_csvs = [
    "10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_abeta.csv",
    "10122021_BaharehNP_ADCortex_EDTA_abeta.csv",
    "10202021_BaharehNP_2997-ADCortex_EDTA_abeta.csv",
    "11162021_Bahareh-3155_EDTA_abeta.csv",
    "12032021-Bahareh_3280_Alz_abeta.csv"
]

abeta_sample_num_dict = {
    "10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_abeta.csv":'1796',
    "10122021_BaharehNP_ADCortex_EDTA_abeta.csv":'3196',
    "10202021_BaharehNP_2997-ADCortex_EDTA_abeta.csv":'2997',
    "11162021_Bahareh-3155_EDTA_abeta.csv":'3155',
    "12032021-Bahareh_3280_Alz_abeta.csv":'3280'
}

abeta_sample_num_dict = {key: [value, 'aBeta'] for key, value in abeta_sample_num_dict.items()}









csvs = iba_csvs + gfap_csvs + olig2_csvs + neun_csvs

sample_num_dicts = [iba_sample_num_dict, gfap_sample_num_dict, olig2_sample_num_dict, neun_sample_num_dict, abeta_sample_num_dict]

sample_num_dict = {}

for d in sample_num_dicts:
    sample_num_dict.update(d)


dfs= []

for csv_file in csvs:
    print(csv_file)
    df = pd.read_csv((folder_path + csv_file))
    print(df.shape)
    df['ImageID'] = sample_num_dict[csv_file][0]
    df['phenotype'] = sample_num_dict[csv_file][1]
    df['ImageID_phenotype'] = df['ImageID'] + '_' + df['phenotype']
    #df['ImageType'] = AD_control_dict[csv_file]
    df['CellID'] = df['cell']
    df['ImageID_CellID'] = df['ImageID'] + '_' + df['CellID'].astype(str)
    df['marker_intensity_csv_filename'] = csv_file

    
    
    dfs.append(df)
    
    

# now also add the abeta

for csv_file in abeta_csvs:
    print(csv_file)
    df = pd.read_csv((folder_path + csv_file))
    
    print(df.shape)
    df['ImageID'] = sample_num_dict[csv_file][0]
    df['phenotype'] = sample_num_dict[csv_file][1]
    df['phenotype'] = df['phenotype'] + df['class']
    df['ImageID_phenotype'] = df['ImageID'] + '_' + df['phenotype']
    #df['ImageType'] = AD_control_dict[csv_file]
    df['CellID'] = df['cell']
    df['ImageID_CellID'] = df['ImageID'] + '_' + df['CellID'].astype(str)
    df['marker_intensity_csv_filename'] = csv_file

    
    dfs.append(df)



combined_df = pd.concat(dfs)


AD_control_dict = {
    '1796': 'Ctl',
    '3196': 'AD',
    '2997': 'AD',
    '1873': 'Ctl',
    '3026': 'Ctl',
    '3155': 'AD',
    '3280': 'AD',
    '3628': 'Ctl'
}

combined_df['ImageType'] = [AD_control_dict[i] for i in combined_df['ImageID']]



## add GM/WM data

In [ ]:

import geopandas as gpd
import geojson
from shapely.geometry import shape, Point

geojson_folder_path = 'GM_WM_Annotations_QP/'
geojson_dict = {
    "1796":"1796_GM-WM Annotations.geojson",
    "1873":"1873_GM-WM Annotations.geojson",
    "2997":"2997_GM-WM Annotations.geojson",
    "3026":"3026_GM-WM Annotations.geojson",
    "3155":"3155_GM-WM Annotations.geojson",
    "3196":"3196_GM-WM Annotations.geojson",
    "3280":"3280_GM-WM Annotations.geojson",
    "3628":"3628_GM-WM Annotations.geojson"
}


def find_department(point):
    for index, row in df.iterrows():
        if not row.geometry.is_valid:
            row.geometry = row.geometry.buffer(0)
        if row.geometry.contains(point):
            return row['classification']['name']


dfs_fixed = []

for sample_filename in list(OrderedDict.fromkeys(combined_df.ImageID)):
    print(sample_filename)
    sub_df = combined_df[combined_df.ImageID == sample_filename]
    geojson_filename = geojson_folder_path + geojson_dict[sample_filename]

    with open(geojson_filename, encoding='UTF-8') as json_file:
        geojson_f = geojson.load(json_file)
        
        df = gpd.GeoDataFrame.from_features(geojson_f["features"])
        
        parents = [find_department(Point(i[0],i[1])) for i in zip(sub_df.x, sub_df.y)]
        sub_df['Parent'] = parents

        category_to_color = {'Grey Matter': 'red', 'White Matter': 'blue', None: 'Gray'}

        color_vector = [category_to_color[category] for category in parents]

        
        plt.scatter(x = sub_df.x, y = sub_df.y, c = color_vector, s = 0.5)
        plt.title(sample_filename)
        plt.show()
        dfs_fixed.append(sub_df)
        
        
        
        
        # we will also find and store the areas of each sample's GM and WM for other parts of the pipeline
        
        df['area'] = [i.area for i in df['geometry']]
        df['zone'] = [i['name'] for i in df['classification']]
        df['area_um'] = ((df['area'] ** 0.5) * 0.3774)**2
        df['area_mm'] = (((df['area'] ** 0.5) * 0.3774)/1000)**2
        df_summed = df.groupby('zone').sum('area_mm')
        df_summed.to_csv('sample_area_files/' + sample_filename + '.csv')
        
        
combined_df = pd.concat(dfs_fixed)
combined_df['ImageID_phenotype_CellID'] = combined_df['ImageID_phenotype'] + '_' + combined_df['CellID'].astype(str)










In [ ]:

spatialCols = ['x','y','area µm²','ImageID','ImageType','Parent','ImageID_CellID', 'cell', 'marker_intensity_csv_filename', 'UMAP_1', 'UMAP_2', 'TSNE_1','TSNE_2','CellID', 'ImageID_phenotype', 'phenotype', 'ImageID_phenotype_CellID', 'None', 'class', 'area_um2', 'perimeter_um', 'cell_circularity', 'cell_solidity', 'a-BetaPSD-95', 'P2Rx7_Park7']
proteinCols = [i for i in combined_df.columns if i not in spatialCols]


spatialNames  = {'x':'spatial_X','y': 'spatial_Y'}

spatDF = combined_df[spatialCols]
spatDF.rename(columns=spatialNames, inplace=True)

protDF = combined_df[proteinCols]

newNames = {col:col.split(':')[0] for col in protDF.columns}
protDF.rename(newNames, inplace=True, axis=1)

adata = ad.AnnData (protDF)
adata.obs = spatDF


imageTypeMapQPv2 = {'2997': 'AD-2',
 '3155': 'AD-1',#Used for classification
 '3196': 'AD-3',
 '3280': 'AD-4',
 '1796': 'Ctl-3',
 '3628': 'Ctl-2',
 '3026': 'Ctl-1', #Used for classification
 '1873': 'Ctl-4'}
adata.obs.loc[:,'ImageIDOLD'] = adata.obs.ImageID
adata.obs.loc[:,'ImageID'] = [imageTypeMapQPv2[i] for i in adata.obs['ImageIDOLD']]

adata.obs.index = adata.obs.ImageID_phenotype_CellID.to_list()


adata.obs['phenotype_translated'] = adata.obs.phenotype.map({'NeuN':'Neurons',
                                                           'GFAP':'Astrocytes',
                                                           'IBA1':'Microglia',
                                                           'Olig2':'Oligodendrocytes',
                                                           'aBetadense':'dense_aBeta',
                                                           'aBetadiffuse':'diffuse_aBeta'
                                                         })


In [ ]:
adata.write_h5ad('fabian_metadata_allcells.h5ad')


### also save individual cell metadatas 

In [ ]:

if not os.path.exists("individual_celltype_metadatas"): 
    os.makedirs("individual_celltype_metadatas") 


for celltype in adata.obs.phenotype_translated.unique():
    print(celltype)
    
    adata_sub = adata[adata.obs.phenotype_translated == celltype]
    adata_sub.write_h5ad('individual_celltype_metadatas/' + celltype + '_metadata.h5ad')
    
    
# also save the dense and diffuse as one object

abeta_data = adata[adata.obs.phenotype_translated.isin(['dense_aBeta', 'diffuse_aBeta'])]
abeta_data.write_h5ad('individual_celltype_metadatas/all_abeta_metadata.h5ad')



# generate metadata for microglia cells

## combine marker intensity csvs


In [ ]:

folder_path = 'marker_intensities/'

csvs = [
'10222021_BaharehNP_1873_CtrlCortex_IBA1_cell1.csv',
 '11162021_Bahareh-3155_EDTA_IBA1_cell1.csv',
 '10202021_BaharehNP_2997-ADCortex_EDTA_IBA1_cell1.csv',
 '12252021_Bahareh-3628-Cntr_IBA1_cell1.csv',
 '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_IBA1_cell1.csv',
 '12032021-Bahareh_3280_Alz_IBA1_cell1.csv',
 '10122021_BaharehNP_ADCortex_EDTA_IBA1_cell1.csv',
 '11092021_BaharehNP_3026CtlCorte_IBA1_cell1.csv'
]





sample_num_dict = {
    '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_IBA1_cell1.csv': "1796",
    '10122021_BaharehNP_ADCortex_EDTA_IBA1_cell1.csv': "3196",
    '10202021_BaharehNP_2997-ADCortex_EDTA_IBA1_cell1.csv': "2997",
    '10222021_BaharehNP_1873_CtrlCortex_IBA1_cell1.csv': "1873",
    '11092021_BaharehNP_3026CtlCorte_IBA1_cell1.csv': "3026",
    '11162021_Bahareh-3155_EDTA_IBA1_cell1.csv': "3155",
    '12032021-Bahareh_3280_Alz_IBA1_cell1.csv': "3280",
    '12252021_Bahareh-3628-Cntr_IBA1_cell1.csv': "3628",
}


AD_control_dict = {
    '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_IBA1_cell1.csv': 'Ctl',
    '10122021_BaharehNP_ADCortex_EDTA_IBA1_cell1.csv': 'AD',
    '10202021_BaharehNP_2997-ADCortex_EDTA_IBA1_cell1.csv': 'AD',
    '10222021_BaharehNP_1873_CtrlCortex_IBA1_cell1.csv': 'Ctl',
    '11092021_BaharehNP_3026CtlCorte_IBA1_cell1.csv': 'Ctl',
    '11162021_Bahareh-3155_EDTA_IBA1_cell1.csv': 'AD',
    '12032021-Bahareh_3280_Alz_IBA1_cell1.csv': 'AD',
    '12252021_Bahareh-3628-Cntr_IBA1_cell1.csv': 'Ctl',
}


morph_data_path = 'morph_data/'
morph_data_columns = ['perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um', 'bi_um']

morph_data_dict = {
    '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_IBA1_cell1.csv': '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_Combined_cell1.csv',
    '10122021_BaharehNP_ADCortex_EDTA_IBA1_cell1.csv': '10122021_BaharehNP_ADCortex_EDTA_Combined_cell1.csv',
    '10202021_BaharehNP_2997-ADCortex_EDTA_IBA1_cell1.csv': '10202021_BaharehNP_2997-ADCortex_EDTA_Combined_cell1.csv',
    '10222021_BaharehNP_1873_CtrlCortex_IBA1_cell1.csv': '10222021_BaharehNP_1873_CtrlCortex_Combined_cell1.csv',
    '11092021_BaharehNP_3026CtlCorte_IBA1_cell1.csv': '11092021_BaharehNP_3026CtlCorte_Combined_cell1.csv',
    '11162021_Bahareh-3155_EDTA_IBA1_cell1.csv': '11162021_Bahareh-3155_EDTA_Combined_cell1.csv',
    '12032021-Bahareh_3280_Alz_IBA1_cell1.csv': '12032021-Bahareh_3280_Alz_Combined_cell1.csv',
    '12252021_Bahareh-3628-Cntr_IBA1_cell1.csv': '12252021_Bahareh-3628-Cntr_Combined_cell1.csv'
}


csv_files_in_which_h2ax_and_gfap_were_switched = ['12252021_Bahareh-3628-Cntr_IBA1_cell1.csv', '12032021-Bahareh_3280_Alz_IBA1_cell1.csv']


dfs= []

for csv_file in csvs:
    print(csv_file)
    df = pd.read_csv((folder_path + csv_file))
    print(df.shape)
    df['ImageID'] = sample_num_dict[csv_file]  
    df['ImageType'] = AD_control_dict[csv_file]
    df['CellID'] = df['cell']
    df['ImageID_CellID'] = df['ImageID'] + '_' + df['CellID'].astype(str)
    df['marker_intensity_csv_filename'] = csv_file
    
    morph_data_csv = morph_data_dict[csv_file]
    morph_data = pd.read_csv((morph_data_path + morph_data_csv))
    morph_data_colsub = morph_data[['cell'] + morph_data_columns]
    morph_data_rowsub = morph_data_colsub[morph_data_colsub.cell.isin(df.cell)]
    
    print(df.shape)
    print(morph_data_rowsub.shape)
    print('')
    
    
    
    
    df = pd.merge(df, morph_data_rowsub, on='cell', how='left')
    
    
    
    # add morph data to matrix
    
    if csv_file in csv_files_in_which_h2ax_and_gfap_were_switched:
        orig_gfap = df['GFAP']
        df['GFAP'] = df['H2A.x']
        df['H2A.x'] = orig_gfap
    
    dfs.append(df)

combined_df = pd.concat(dfs)



## add relevant metadata

### add GM/WM

In [ ]:
import geopandas as gpd
import geojson
from shapely.geometry import shape, Point

geojson_folder_path = 'GM_WM_Annotations_QP/'
geojson_dict = {
    "1796":"1796_GM-WM Annotations.geojson",
    "1873":"1873_GM-WM Annotations.geojson",
    "2997":"2997_GM-WM Annotations.geojson",
    "3026":"3026_GM-WM Annotations.geojson",
    "3155":"3155_GM-WM Annotations.geojson",
    "3196":"3196_GM-WM Annotations.geojson",
    "3280":"3280_GM-WM Annotations.geojson",
    "3628":"3628_GM-WM Annotations.geojson"
}


def find_department(point):
    for index, row in df.iterrows():
        if not row.geometry.is_valid:
            row.geometry = row.geometry.buffer(0)
        if row.geometry.contains(point):
            return row['classification']['name']


dfs_fixed = []

for sample_filename in list(OrderedDict.fromkeys(combined_df.marker_intensity_csv_filename)):
    print(sample_filename)
    sub_df = combined_df[combined_df.marker_intensity_csv_filename == sample_filename]
    sub_ImageID = sub_df.ImageID.tolist()[0]
    geojson_filename = geojson_folder_path + geojson_dict[sub_ImageID]

    with open(geojson_filename, encoding='UTF-8') as json_file:
        geojson_f = geojson.load(json_file)
        df = gpd.GeoDataFrame.from_features(geojson_f["features"])
        parents = [find_department(Point(i[0],i[1])) for i in zip(sub_df.x, sub_df.y)]
        sub_df['Parent'] = parents

        category_to_color = {'Grey Matter': 'red', 'White Matter': 'blue', None: 'Gray'}

        color_vector = [category_to_color[category] for category in parents]

        
        plt.scatter(x = sub_df.x, y = sub_df.y, c = color_vector, s = 0.5)
        plt.title(sample_filename)

        dfs_fixed.append(sub_df)
        
combined_df = pd.concat(dfs_fixed)




### make adata

In [ ]:


spatialCols = ['x','y','area µm²','ImageID','ImageType','Parent','ImageID_CellID', 'cell', 'marker_intensity_csv_filename', 'UMAP_1', 'UMAP_2', 'TSNE_1','TSNE_2','CellID', 'perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um', 'bi_um']
proteinCols = [i for i in combined_df.columns if i not in spatialCols]


spatialNames  = {'x':'spatial_X','y': 'spatial_Y'}

spatDF = combined_df[spatialCols]
spatDF.rename(columns=spatialNames, inplace=True)

protDF = combined_df[proteinCols]

newNames = {col:col.split(':')[0] for col in protDF.columns}
protDF.rename(newNames, inplace=True, axis=1)

adata = ad.AnnData (protDF)
adata.obs = spatDF


imageTypeMapQPv2 = {'2997': 'AD-2',
 '3155': 'AD-1',#Used for classification
 '3196': 'AD-3',
 '3280': 'AD-4',
 '1796': 'Ctl-3',
 '3628': 'Ctl-2',
 '3026': 'Ctl-1', #Used for classification
 '1873': 'Ctl-4'}
adata.obs.loc[:,'ImageIDOLD'] = adata.obs.ImageID
adata.obs.loc[:,'ImageID'] = [imageTypeMapQPv2[i] for i in adata.obs['ImageIDOLD']]

adata.obs.index = adata.obs.ImageID_CellID.to_list()


### save object

In [ ]:
# note:

# data must be batch corrected and then scaled prior to clustering

# switch x and y

# old_x = copy.deepcopy(adata.obs.spatial_X)
# adata.obs.spatial_X = copy.deepcopy(adata.obs.spatial_Y)
# adata.obs.spatial_Y = old_x

adata.write_h5ad('fabian_metadata.h5ad')


# add distances

## load libs, functions and data

In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
#from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import re
import imagecodecs
import h5py
from collections import OrderedDict


#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors
    
def crop_image(matrix, center_coordinate, buffer_size):
    # Extract dimensions of the original matrix
    height, width = matrix.shape

    # Extract the coordinates of the center
    center_x, center_y = center_coordinate

    print(center_x)
    
    
    # Calculate the bounding box for the cropped image
    start_x = max(0, center_x - buffer_size)
    end_x = min(height, center_x + buffer_size)
    start_y = max(0, center_y - buffer_size)
    end_y = min(width, center_y + buffer_size)

    # Calculate the cropped image
    cropped_image = matrix[start_x:end_x, start_y:end_y]

    return cropped_image




# load in sample all cells adata

In [ ]:

#adata = anndata.read_h5ad('/home/users/browneld/lab/CODEX/codex-cns-devzone/figure-notebooks-playground/generate_fabian_seg_metadata/fabian_metadata_allcells.h5ad')

adata = anndata.read_h5ad('fabian_metadata_allcells.h5ad')

## generate x/y coordinates for non-nucleated features


In [ ]:

# generate x/y coordinates for non-nucleated features

df_dict = {
'image_files':['image_data/11092021_BaharehNP_3026CtlCorte.qptiff', 'image_data/11162021_Bahareh-3155_EDTA.qptiff', 'image_data/12032021-Bahareh_3280_Alz.qptiff', 'image_data/12252021_Bahareh-3628-Cntr.qptiff', 'image_data/10222021_BaharehNP_1873_CtrlCortex.qptiff', 'image_data/10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6.qptiff', 'image_data/10122021_BaharehNP_ADCortex_EDTA_3196.qptiff', 'image_data/10202021_BaharehNP_2997-ADCortex_EDTA.qptiff'],
'image_nums':[3026, 3155, 3280, 3628, 1873, 1796, 3196, 2997],
'thresholds':[{'DAPI_1':30, 'Neurofilament':10, 'Vimentin':20, 'Claudin5':24, 'SMA':25, 'A-Beta':30, 'ApoE':60, 'GFAP':30, 'Olig-2':30, 'PSD95':30, 'Synaptophysin':20, 'NeuN':25, 'H2A.x':25, 'CollagenIV':30, 'PCNA':100, 'CD3e':70, 'CD4':80, 'MAP2':20},
             {'DAPI_1':25, 'Neurofilament':20, 'Vimentin':10, 'Claudin-5':60, 'SMA':20, 'a-Beta':18, 'GFAP':33, 
                     'Olig2':26, 'Synaptophysin':20,'ApoE':49, 'IBA1':30, 'H2A.x':15, 'CollagenIV':20, 'PCNA':175, 'MAP2':30}, 
             {'DAPI_1':25, 'Neurofilament':35, 'Vimentin':25, 'Claudin-5':20, 'SMA':20, 'a-Beta':15, 'Olig2':20, 'PSD-95':15, 'ApoE':40, 'NeuN':15, 'H2A.x':35, 'CollagenIV':10, 'PCNA':115, 'MAP2':15},
             {'DAPI_1':30, 'Neurofilament':20, 'Vimentin':30, 'Claudin-5':20, 'SMA':24, 'a-Beta':255, 'Olig2':60, 'PSD-95':20,'ApoE':80, 'CD8':254, 'NeuN':10, 'H2A.x':5, 'CollagenIV':30, 'PCNA':125, 'CD3e':47, 'CD4':50, 'MAP2':20}, {'DAPI_1':30, 'Neurofilament':25, 'Vimentin':20, 'Claudin5':80, 'SMA':20, 'A-Beta':255,'ApoE':60, 'GFAP':35, 'Synaptophysin':10, 'CD8':200, 'NeuN':20, 'H2A.x':55, 'CollagenIV':25, 'PCNA':145, 'CD3e':220, 'MAP2':25}, {'DAPI_1':30, 'Neurofilament':28, 'Vimentin':30, 'Claudin5':43, 'SMA':30, 'A-Beta':30, 'GFAP':30, 'ApoE':60, 'NeuN':25, 'H2A.x':46, 'CollagenIV':30, 'PCNA':103, 'CD3e':116, 'MAP2':40}, {'DAPI_1':35, 'Neurofilament':25, 'Vimentin':40, 'Claudin5':36, 'SMA':30, 'A-Beta':26, 'GFAP':25, 'ApoE':40, 'CD8':250, 'P2Rx7(Park7)':160, 'S100B':80, 'NeuN':30, 'H2A.x':26, 'CollagenIV':22, 'PCNA':230, 'CD3e':250, 'CD4':250, 'MAP2':42}, {'DAPI_1':25, 'Neurofilament':15, 'Vimentin':15, 'Claudin5':18, 'SMA':10, 'A-Beta':15, 'ApoE':40, 'GFAP':20, 'Synaptophysin':27, 'P2Rx7(Park7)':157, 'S100B':13, 'NeuN':15, 'H2A.x':25, 'CollagenIV':10, 'PCNA':117, 'MAP2':30}
],
'nn_structures':[['Claudin5', 'A-Beta', 'ApoE'], ['Claudin-5', 'a-Beta', 'ApoE'], ['Claudin-5', 'a-Beta', 'ApoE'], ['Claudin-5', 'a-Beta', 'ApoE'], ['Claudin5', 'A-Beta', 'ApoE'], ['Claudin5', 'A-Beta', 'ApoE'], ['Claudin5', 'A-Beta', 'ApoE'], ['Claudin5', 'A-Beta', 'ApoE']],
'nn_structure_names':[['vasculature', 'aBeta', 'ApoE']] * 8,
'noise_pixel_thresh':[[100,1404, 1404]] * 8,
'GM_WM_mask':['GM_WM_Annotations_QP/3026_GM-WM Annotations.geojson', 'GM_WM_Annotations_QP/3155_GM-WM Annotations.geojson', 'GM_WM_Annotations_QP/3280_GM-WM Annotations.geojson', 'GM_WM_Annotations_QP/3628_GM-WM Annotations.geojson', 'GM_WM_Annotations_QP/1873_GM-WM Annotations.geojson', 'GM_WM_Annotations_QP/1796_GM-WM Annotations.geojson', 'GM_WM_Annotations_QP/3196_GM-WM Annotations.geojson', 'GM_WM_Annotations_QP/2997_GM-WM Annotations.geojson']
}

# orig vasc thresh was 400

binar_matrices = {}

image_df = pd.DataFrame(df_dict)

for col in image_df.index:
    row = image_df.iloc[col,:]
    (all_pages,shapes) = tiff_in(row['image_files'])
    markerDict = channelNames(row['image_files'])
    markerDict_reverse = inv_map = {v: k for k, v in markerDict.items()}
    
    
    
    
    for geneind in range(len(row['nn_structures'])):
        
        
        gene = row['nn_structures'][geneind]
        
        

        
        structure_name = row['nn_structure_names'][geneind]
        orig_Img = all_pages[0][markerDict_reverse[gene]]
        print(gene)
        imshow(orig_Img)
        plt.show()
        crop_orig = crop_image(orig_Img, center_coordinate=[1805, 3408], buffer_size=50)
        imshow(crop_orig)
        
        plt.title(str(row['image_nums']) + gene)
        plt.show()
        
        
        tMin = row['thresholds'][gene]
        flat = orig_Img.flatten()
        flat = flat[flat >= tMin]
        a_max = min(np.percentile(flat, q = 99), tMin + 30)
        
        inImg_binar = np.clip(a=orig_Img,a_min=0, a_max = a_max)
        inImg_binar[inImg_binar < tMin] = 0
        inImg_binar[inImg_binar > 0] = 1
        binar_matrices[structure_name + '_' + str(row['image_nums'])] = [inImg_binar, gene, structure_name, structure_name + '_' + str(row['image_nums']), str(row['image_nums']), row['noise_pixel_thresh'][geneind], row['GM_WM_mask']]
        
        
        
        

    
    
    
    # load appropriate image





In [ ]:

import h5py


big_griddles = {}


from skimage import io, color, measure


for image_key in binar_matrices.keys():
    print(image_key)
    
    image = binar_matrices[image_key][0]
    id = binar_matrices[image_key][3]
    structure_name = binar_matrices[image_key][2]
    GM_WM_mask = binar_matrices[image_key][6]
    filenum = binar_matrices[image_key][4]
    
    # Define your original matrix
    original_matrix = image

    # Get the shape of the original matrix
    num_rows, num_cols = original_matrix.shape

    # label connected components 
    labeled_image = measure.label(original_matrix)
    
    # save segmentation
    
    
    import h5py
    import numpy as np
    
    
    
    seg_dict = {}
    seg_dict['matrix'] = np.transpose(labeled_image)
    
#     def dict_to_hdf5(group, data):
#         for key, value in data.items():
#             if isinstance(value, dict):
#                 subgroup = group.create_group(key)
#                 dict_to_hdf5(subgroup, value)
#             else:
#                 group[key] = value

#     with h5py.File('stored_segmentations/' + structure_name + '_' + filenum + '.h5', 'w') as f:
#         dict_to_hdf5(f, seg_dict)
    
    
    
    
    
    
    
    # Find the properties of labeled regions (blobs)
    blob_properties = measure.regionprops(labeled_image)


    # Plot the original image
    plt.imshow(image[2000:4000,2000:4000], cmap='gray')

    # for region in blob_properties:
    #     minr, minc, maxr, maxc = region.bbox
    #     rect = plt.Rectangle((minc, minr), maxc - minc, maxr - minr, fill=False, edgecolor='red', linewidth=2)
    #     plt.gca().add_patch(rect)
    # plt.show()
    # jdklf
    
    # Extract the centroids (x, y coordinates) of the detected blobs
    blob_centroids = [(int(prop.centroid[0]), int(prop.centroid[1])) for prop in blob_properties]
    blob_areas = [int(prop.area) for prop in blob_properties]
    blob_densities = [blob.area / blob.bbox_area for blob in blob_properties]

    # threshold centroids by area
    noise_pixel_thresh = binar_matrices[image_key][5]
    blob_centroids_thresholded = [i[0] for i in zip(blob_centroids, np.array(blob_areas) > noise_pixel_thresh) if i[1]]
    blob_densities_thresholded = [i[0] for i in zip(blob_densities, np.array(blob_areas) > noise_pixel_thresh) if i[1]]

    
    
    # plot centroids

    ig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(original_matrix)
    
    for centroid in blob_centroids_thresholded:
        ax.plot(centroid[1], centroid[0], 'ro', markersize=0.1)  # Note the (y, x) order for plotting
        
    ax.set_title(image_key)
    plt.show()
    
    
    
    gooby = plt.imshow(image[2000:4000,2000:4000])
    big_centroid_df = pd.DataFrame(blob_centroids_thresholded, columns=['X','Y'])
    #big_centroid_df['density'] = blob_densities_thresholded
    
    for centroid in big_centroid_df[(big_centroid_df['X'] > 2000) & (big_centroid_df['X'] < 4000) & (big_centroid_df['Y'] > 2000) & (big_centroid_df['Y'] < 4000)].index:
        row = big_centroid_df.iloc[centroid]
        
        plt.plot(row['Y'] - 2000, row['X'] - 2000, 'ro', markersize=1)  # Note the (y, x) order for plotting
    plt.title(image_key)
    plt.show()
    
    big_griddles[id] = [blob_centroids_thresholded, image, GM_WM_mask, structure_name, filenum]

    
    
    

## assign GM/WM

In [ ]:
import geopandas as gpd
import geojson
from shapely.geometry import shape, Point

def find_department(point):
    for index, row in df.iterrows():
        if not row.geometry.is_valid:
            row.geometry = row.geometry.buffer(0)
        if row.geometry.contains(point):
            return row['classification']['name']

big_df_dict = {'spatial_X':[], 'spatial_Y':[], 'Parent':[], 'filenum':[], 'phenotype':[]}

for griddlegroup_name in big_griddles:
    print(griddlegroup_name)
    griddlegroup = big_griddles[griddlegroup_name]
    griddle = griddlegroup[0]
    filenum = griddlegroup[4]
    image = big_griddles[griddlegroup_name][1]
    GM_WM_mask = griddlegroup[2]
    print(GM_WM_mask)
    structure_name = griddlegroup[3]
    
    
    with open(GM_WM_mask, encoding='UTF-8') as json_file:
        geojson_f = geojson.load(json_file)
        df = gpd.GeoDataFrame.from_features(geojson_f["features"])
        griddleparents = [find_department(Point(i[1],i[0])) for i in griddle]
    big_df_dict['spatial_X'].extend([i[1] for i in griddle])
    big_df_dict['spatial_Y'].extend([i[0] for i in griddle])
    big_df_dict['Parent'].extend(griddleparents)
    big_df_dict['filenum'].extend([filenum]*len(griddle))
    big_df_dict['phenotype'].extend([structure_name]*len(griddle))
    
        
        
non_nuc_df = pd.DataFrame(big_df_dict)       
    

### load fabian metadata and clusters

In [ ]:

adata = anndata.read_h5ad('fabian_metadata.h5ad')

adata_meta = adata.obs


print(adata.obs.shape)
print(adata_meta.shape)

### generate custom distances

In [ ]:
from scipy.spatial.distance import cdist


samples = set(non_nuc_df.filenum)
phenotypes = set(non_nuc_df.phenotype)


for phenotype in phenotypes:
    distance_dict = {}
    for sample_num in samples:
        print(phenotype)
        print(sample_num)
        
        sub_non_nuc = non_nuc_df[(non_nuc_df.filenum == sample_num) & (non_nuc_df.phenotype == phenotype)]
        sub_metadata = adata_meta[adata_meta.ImageIDOLD == sample_num]
        
        print(sub_metadata.shape)
        
        set1 = zip(sub_non_nuc.spatial_X, sub_non_nuc.spatial_Y)
        set1 = [list(item) for item in set1]
        

        set2 = zip(sub_metadata.spatial_X, sub_metadata.spatial_Y)
        set2 = [list(item) for item in set2]

        # Calculate pairwise distances
        distances = cdist(set2, set1)
        # Find the minimum distance for each point in set1
        min_distances = np.min(distances, axis=1)
        
        distance_dict_section = dict(zip(sub_metadata.ImageID_CellID, min_distances))
        distance_dict.update(distance_dict_section)
    adata_meta['custom_distance_to_nearest_' + phenotype] = [distance_dict[i] for i in adata_meta.ImageID_CellID]
        




### generate normalseg distances

In [ ]:

adata_allcells = anndata.read_h5ad('fabian_metadata_allcells.h5ad')
phenotypes_to_profile = [['Astrocytes'], ['Neurons'], ['Oligodendrocytes'], ['dense_aBeta'], ['diffuse_aBeta'], ['dense_aBeta', 'diffuse_aBeta']]

adata_allcells_meta = adata_allcells.obs


from scipy.spatial.distance import cdist


samples = set(adata_meta.ImageIDOLD)
phenotypes = phenotypes_to_profile


for phenotype in phenotypes:
    distance_dict = {}
    for sample_num in samples:
        print(phenotype)
        print(sample_num)
        
        sub_allcells_meta = adata_allcells_meta[(adata_allcells_meta.ImageIDOLD == sample_num) & (adata_allcells_meta.phenotype_translated.isin(phenotype))]
        
        
        
        sub_metadata = adata_meta[adata_meta.ImageIDOLD == sample_num]
        
        
        
        print(sub_metadata.shape)
        print(sub_allcells_meta.shape)
        
        if sub_allcells_meta.shape[0] == 0:
            distance_dict_section = dict(zip(sub_metadata.ImageID_CellID, [np.NaN]*len(sub_metadata.ImageID_CellID)))
            distance_dict.update(distance_dict_section)
            continue
        
        set1 = zip(sub_allcells_meta.spatial_X, sub_allcells_meta.spatial_Y)
        set1 = [list(item) for item in set1]
        

        set2 = zip(sub_metadata.spatial_X, sub_metadata.spatial_Y)
        set2 = [list(item) for item in set2]

        # Calculate pairwise distances
        distances = cdist(set2, set1)
        # Find the minimum distance for each point in set1
        min_distances = np.min(distances, axis=1)
        
        distance_dict_section = dict(zip(sub_metadata.ImageID_CellID, min_distances))
        distance_dict.update(distance_dict_section)
        
        
    adata_meta['fabseg_distance_to_nearest_' + ''.join(phenotype)] = [distance_dict[i] for i in adata_meta.ImageID_CellID]
        



## save object

In [ ]:
adata.obs = adata_meta

adata.write_h5ad('fabian_metadata.h5ad')

# generate some basic celltype abundance stats for paper

In [ ]:
import anndata 


all_cells_adata = anndata.read_h5ad('fabian_metadata_allcells.h5ad')
all_cells_adata_noabeta = all_cells_adata[all_cells_adata.obs.phenotype_translated.isin(['dense_aBeta', 'diffuse_aBeta']) == False]


print('total cells segmented:')
print(len(all_cells_adata_noabeta.obs.phenotype_translated))

print('breakdown of cell types:')
print(collections.Counter(all_cells_adata_noabeta.obs.phenotype_translated))

print('breakdown of WM/GM')
print(collections.Counter(all_cells_adata_noabeta.obs.Parent))


just_mg_adata = anndata.read_h5ad('fabian_metadata.h5ad')

print('number of iba cells that went into clustering')
print(len(just_mg_adata))



